In [ ]:
### conda activate [] environment with scvi-tools

import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from scvi.external import MRVI
import os
import seaborn as sns
import json
from matplotlib.lines import Line2D

matplotlib.rcParams['pdf.fonttype'] = 42

import sys
sys.path.append('../../1_figure_CL_proof_of_concept/code/')
import utils_00 as gf_utils
large_data_dir = gf_utils.large_data_dir


In [ ]:
def rename_cell_types(ad):
    ad.obs.loc[ad.obs['cell_type'] == 'Classical monocyte 2 (non-HSPC)','cell_type'] = 'Classical monocyte (non-HSPC)'
    ad = ad[ad.obs['cell_type'] != 'Monocyte/Platelet doublet (non-HSPC)'].copy()
    ad.obs.loc[~ad.obs['cell_type'].str.contains('(non-HSPC)'), 'coarse_cell_type'] = 'HSPC'
    ad.obs['cell_type'] = ad.obs['cell_type'].str.replace(r'\s*\(non-HSPC\)', '', regex=True)
    ad.obs['cell_type'] = ad.obs['cell_type'].str.replace(r'\s*\[tentative\]', '', regex=True)
    ad.obs['cell_type'] = ad.obs['cell_type'].str.replace('MPP (myeloid-primed)','MPP')
    return ad


## load full mrvi object just to get cell type groups
adata = sc.read_h5ad(large_data_dir + 'MPN_WTA/homozygous_JAK2_mrvi_patient_as_nuisance.h5ad')
adata = rename_cell_types(adata)

## define groups of cell types - HSC, monocytes, early MK

cell_type_df = pd.DataFrame(adata.obs['cell_type'].unique(), columns = ['cell_type'])
cell_type_df.loc[cell_type_df['cell_type'].isin(['HSC']), 'cluster'] = 'HSC'
cell_type_df.loc[cell_type_df['cell_type'].str.contains('lassical monocyte'), 'cluster'] = 'monocyte' ### classical and non-classical but not activated monocytes
cell_type_df = cell_type_df.dropna().sort_values('cluster')

del adata

In [ ]:
os.environ["JAX_PLATFORMS"] = "cpu" ### models are already trained so GPU not needed

DE = pd.DataFrame()

for sample in ['BC001_1plex','BC001_4plex','BC002_1','BC003_1','BC003_4plex','BC014_1']:

    temp_adata = sc.read_h5ad(large_data_dir + 'MPN_WTA/' + sample + '_mrvi.h5ad')
    temp_adata = rename_cell_types(temp_adata)

    model = MRVI.load(large_data_dir + 'MPN_WTA/' + sample + '_mrvi_model/', temp_adata)

    sample_cov_keys = ["genotype"]  # Replace with your sample covariate of interest
    model.sample_info["genotype"] = model.sample_info["genotype"].cat.reorder_categories(
        ["wt", "mutated"]
    )  # Reorder categories such that the coefficient corresponds to genotype
    de_res = model.differential_expression(
        sample_cov_keys=sample_cov_keys,
        store_lfc=True,
    )
    for cell_cluster in cell_type_df['cluster'].unique():
        cell_types = cell_type_df.loc[cell_type_df['cluster'] == cell_cluster, 'cell_type'].tolist()
        cell_idxs = temp_adata[(temp_adata.obs["cell_type"].isin(cell_types))].obs.index
        if len(cell_idxs) > 20:
            DE = DE.merge(de_res.sel(cell_name=cell_idxs, covariate="genotype_mutated").mean(dim="cell_name").lfc.to_pandas().rename(str(cell_cluster) + '_' + sample), left_index=True, right_index=True, how='outer')


In [ ]:
# Extract sample IDs from column names
def extract_sample_id(col_name):
    for sample in ['BC001_1-plex','BC001_4-plex','BC002_1','BC003_1','BC003_4-plex','BC014_1']:
        if col_name.endswith('_' + sample):
            return sample
    return 'Unknown'

In [ ]:
plot_dfs = {}

for cell_type in cell_type_df['cluster'].unique():
    sub_DE = DE[DE.columns[DE.columns.str.contains(cell_type)]]
    sub_DE = sub_DE.loc[sub_DE.notna().sum(axis=1) > 3] ### require that gene is present in at least 3 samples

    n_genes = len(sub_DE)
    if n_genes == len(sub_DE):
        top_genes = sub_DE.median(axis=1).sort_values(ascending=False).head(n_genes)
        gene_list = top_genes
    else:
        top_genes = sub_DE.median(axis=1).sort_values(ascending=False).head(n_genes)
        bottom_genes = sub_DE.median(axis=1).sort_values(ascending=True).head(n_genes).sort_values(ascending=False)
        gene_list = pd.concat([top_genes,bottom_genes])

    # Create a long-format dataframe for plotting
    plot_data = []
    for gene in gene_list.index:
        for col in sub_DE.columns:
            sample_id = extract_sample_id(col)
            plot_data.append({
                'gene': gene,
                'expression': sub_DE.loc[gene, col],
                'sample_id': sample_id,
                'column': col
            })

    plot_df = pd.DataFrame(plot_data)
    
    top_genes.to_csv('../output/' + cell_type + '_jak2_de_genes.csv')

    plot_dfs[cell_type] = plot_df.copy()
    DE[DE.columns[DE.columns.str.contains(cell_type)]].to_csv('../output/' + cell_type + '_jak2_de_individual_patients.csv')

In [ ]:
with open(f'../data/h.all.v2026.1.Hs.txt') as f:
    genes_by_term = json.load(f)

cell_type_terms = {}
cell_type_terms['HSC'] = ['MYC_TARGETS_V1','INTERFERON_GAMMA_RESPONSE','COAGULATION','E2F_TARGETS']
cell_type_terms['monocyte'] = ['INTERFERON_ALPHA_RESPONSE','COMPLEMENT','COAGULATION','INTERFERON_GAMMA_RESPONSE']


In [ ]:
all_terms = sorted(set(t for terms in cell_type_terms.values() for t in terms))
cmap = plt.cm.tab10 if len(all_terms) <= 10 else plt.cm.tab20
term_colors = {term: cmap(i / len(all_terms)) for i, term in enumerate(all_terms)}

for cell_type in cell_type_terms.keys():

    plot_df = plot_dfs[cell_type]

    fig, ax = plt.subplots(figsize=(6, 4))

    # Grey background for all points
    ax.scatter(plot_df['gene'], plot_df['expression'],
               color='grey', alpha=0.8, s=30, zorder=1)

    # Collect all term-colored points, then shuffle
    colored_rows = []
    for term in cell_type_terms[cell_type]:
        term_genes = genes_by_term['HALLMARK_' + term]['geneSymbols']
        mask = plot_df['gene'].isin(term_genes)
        rows = plot_df[mask].copy()
        rows['_color'] = [term_colors[term]] * len(rows)
        colored_rows.append(rows)

    colored_df = pd.concat(colored_rows).sample(frac=1, random_state=42)
    ax.scatter(colored_df['gene'], colored_df['expression'],
               color=colored_df['_color'].tolist(), alpha=1, s=30, zorder=2)

    handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=term_colors[t],
                       markersize=6, label=t) for t in cell_type_terms[cell_type]]
    ax.legend(handles=handles, fontsize=8, loc='upper right', framealpha=0.9)

    ax.set_xlabel('Genes', fontsize=12)
    ax.set_ylabel('Log Fold Change (MrVI)', fontsize=12)
    ax.set_xticklabels([])
    ax.set_xticks([])
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_ylim(-1, 2)

    plt.tight_layout()
    plt.savefig('../plots/' + cell_type + '_jak2_de_genes_scatter.pdf')

In [ ]:
for cell_type in ['monocyte','HSC']:
    # Prepare heatmap_df
    heatmap_df = (
        plot_dfs[cell_type]
        .groupby('gene')['expression']
        .median()
        .sort_values(ascending=False)
        .reset_index()
    )

    heatmap_data = heatmap_df['expression'].values.reshape(1, -1)
    print(heatmap_df['expression'].max(), heatmap_df['expression'].min())
    # Create the heatmap
    plt.figure(figsize=(6, 0.5))
    sns.heatmap(heatmap_data, 
                cmap='RdBu_r',  # Reversed Blue to red colormap
                cbar=True,     # Hide color bar
                center=0,      # Center the colormap at 0
                xticklabels=False,  # Hide x-axis labels since there are too many genes
                vmin=-0.5, vmax=0.5)
    plt.savefig('../plots/' + cell_type + '_jak2_de_genes_median_heatmap.pdf', bbox_inches='tight')